<a href="https://colab.research.google.com/github/UH-RMDS-DataChallenge-2026/data-science-challenge/blob/main/individual_work/24067150.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---- Core libraries ----
import warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- scikit-learn: pipelines, pre-processing, models, metrics ----
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, log_loss, accuracy_score, brier_score_loss,
                             precision_score, recall_score, f1_score,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")                  # keep the output readable
sns.set_theme(style="whitegrid")                   # consistent plot style
pd.set_option("display.max_columns", 100)          # show all columns in tables

RANDOM_STATE = 42                                  # one seed used everywhere -> reproducible results
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

## 8. Results, ranking and diagnostics

In [ ]:
# Every (model, decision rule) combination ranked by mean cross-validated cost per person
res = pd.DataFrame(results)
rule_cols = ["cost | fixed 0.5", "cost | global threshold", "cost | personal threshold"]
long = res.melt(id_vars=["model", "fold"], value_vars=rule_cols, var_name="rule", value_name="cost")
long["rule"] = long["rule"].str.replace("cost | ", "", regex=False)
ranking = long.groupby(["model", "rule"])["cost"].agg(["mean", "std"]).sort_values("mean").reset_index()
ranking["saving vs send-to-nobody ($/person)"] = baselines["Send to nobody"] - ranking["mean"]
ranking["% of maximum possible saving"] = 100 * ranking["saving vs send-to-nobody ($/person)"] / (
    baselines["Send to nobody"] - baselines["Perfect knowledge (lower bound)"])
display(ranking.head(15).round(2))

In [ ]:
# Compact model table: discrimination, calibration, accuracy and cost under each rule
model_table = res.groupby("model")[["AUC", "log-loss", "accuracy@0.5", "precision (personal)",
                                    "recall (personal)", "F1 (personal)"] + rule_cols].mean()
model_table = model_table.sort_values("cost | personal threshold").round(4)
display(model_table)

# Visual comparison of the decision rules for every model (mean +/- std over folds)
plot_df = long.copy()
order = model_table.index.tolist()
plt.figure(figsize=(11, 8))
sns.barplot(data=plot_df, y="model", x="cost", hue="rule", order=order, errorbar="sd")
for label, val, col in [("Send to nobody", baselines["Send to nobody"], "red"),
                        ("Send to everybody", baselines["Send to everybody"], "orange"),
                        ("Perfect knowledge", baselines["Perfect knowledge (lower bound)"], "green")]:
    plt.axvline(val, ls="--", color=col, label=label)
plt.xlim(baselines["Perfect knowledge (lower bound)"] - 40, baselines["Send to nobody"] + 20)
plt.xlabel("Cost per person ($) – lower is better"); plt.ylabel("")
plt.title("Town cost by model and decision rule (5-fold CV)"); plt.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

**Interpretation.** The **decision rule matters more than the model**: for every model, the personal (risk-based) threshold gives the lowest cost, beating the standard 0.5 cut-off. Differences between the best models are small compared with the fold-to-fold variation (± std), so the ranking among the top models is uncertain, while the advantage of the personal threshold is consistent. The single Decision Tree has clearly the lowest AUC and worst calibration, showing the benefit of ensembles; the personal rule partly compensates for this in cost.

In [ ]:
# Is the personal-threshold advantage consistent? Paired comparison fold by fold
paired = res.assign(gain=res["cost | fixed 0.5"] - res["cost | personal threshold"])
display(paired.pivot(index="model", columns="fold", values="gain").round(2)
        .assign(mean=lambda d: d.mean(axis=1)).sort_values("mean", ascending=False))
print("Folds where the personal threshold beats 0.5:",
      f"{(paired['gain'] > 0).sum()} / {len(paired)}")

### Final check on a held-out 20 %
Cross-validation reuses all data for model selection. As an independent check, we train on 80 % and evaluate the top candidates once on a **held-out 20 %** they have never seen (thresholds come from the assumptions, not from the holdout).

In [ ]:
X_dev, X_hold, y_dev, y_hold, h_dev, h_hold, idx_dev, idx_hold = train_test_split(
    X_eng, y, h_train, np.arange(len(y)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
raw_dev, raw_hold = X_raw.iloc[idx_dev], X_raw.iloc[idx_hold]

def model_and_data(name):
    """Return the pipeline and the matching feature set (original or engineered)."""
    if name in models: return models[name], (X_dev, X_hold)
    if name in tuned:  return tuned[name], (X_dev, X_hold)
    kwargs, Xset = experiments[name]
    return logreg(**kwargs), ((raw_dev, raw_hold) if Xset is X_raw else (X_dev, X_hold))

hold_rows = [{"model": "Send to nobody", "holdout cost": cost_per_person(y_hold, np.zeros(len(y_hold)), h_hold)},
             {"model": "Send to everybody", "holdout cost": cost_per_person(y_hold, np.ones(len(y_hold)), h_hold)}]
for name in model_table.index[:8]:
    mdl, (Xd, Xh) = model_and_data(name)
    p_h = clone(mdl).fit(Xd, y_dev).predict_proba(Xh)[:, 1]
    hold_rows.append({"model": name, "AUC": roc_auc_score(y_hold, p_h),
                      "holdout cost": cost_per_person(y_hold, p_h > personal_threshold(h_hold), h_hold),
                      "holdout cost @0.5": cost_per_person(y_hold, p_h > 0.5, h_hold),
                      "CV cost": model_table.loc[name, "cost | personal threshold"]})
display(pd.DataFrame(hold_rows).sort_values("holdout cost").round(3))

**Interpretation.** The holdout confirms the cross-validation picture: every model beats both baselines on unseen people, and the personal threshold beats the 0.5 cut-off. Holdout costs differ slightly from CV costs because the holdout people differ (sampling noise), which is why we rank by the 5-fold CV average.

In [ ]:
# Diagnostics for the best model: confusion matrix (personal rule), ROC curve and calibration
best_name = model_table.index[0]
p_best = OOF[best_name]
send_best = (p_best > personal_threshold(h_train)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(y, send_best, cmap="Blues", values_format="d", ax=axes[0], colorbar=False)
axes[0].set_xticklabels(["Not sent", "Sent"]); axes[0].set_yticklabels(["Refuses", "Accepts"])
axes[0].set_xlabel("Our decision"); axes[0].set_ylabel("Actual answer")
axes[0].set_title(f"{best_name}\nconfusion matrix (personal thresholds)")
for name in model_table.index[:3]:
    RocCurveDisplay.from_predictions(y, OOF[name], name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="grey"); axes[1].set_title("ROC curves (out-of-fold)")
for name in model_table.index[:3]:
    frac_pos, mean_pred = calibration_curve(y, OOF[name], n_bins=10)
    axes[2].plot(mean_pred, frac_pos, marker="o", label=name)
axes[2].plot([0, 1], [0, 1], "--", color="grey", label="Perfect calibration")
axes[2].set_xlabel("Predicted probability"); axes[2].set_ylabel("Observed acceptance rate")
axes[2].set_title("Calibration (reliability) curves"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{best_name}: accuracy@0.5 = {accuracy_score(y, p_best > 0.5):.3f} | "
      f"accuracy with personal thresholds = {accuracy_score(y, send_best):.3f} | "
      f"Brier score = {brier_score_loss(y, p_best):.4f}")

**Interpretation.** The confusion matrix shows the rule's deliberate trade-off: many people who would accept are *not* sent a vaccine – mostly young, healthy people for whom a vaccine costs more ($300) than the hospital risk it removes. The calibration curves lie close to the diagonal, so the predicted probabilities can be trusted – a requirement for the personal-threshold rule. Accuracy with personal thresholds is *lower* than at 0.5 **by design**: the rule accepts more cheap mistakes (unused vaccines for high-risk people) to avoid expensive ones (unprotected high-risk people who wanted a vaccine). Accuracy treats all errors as equal; the town's budget does not.